# 문제 1-2 : 2단계 Multi Chain 만들기 - 여행지 정보 시스템

- **1단계 체인**: 여행지(도시/국가) → 대표 명소 1가지
- **2단계 체인**: 추천 명소 → 상세 정보(역사, 특징, 방문 팁)
- `ChatPromptTemplate` 에 `"system"`, `"user"` 메시지 지정
- 두 체인을 LCEL 로 연결하고, 각 단계의 결과를 모두 출력

### 공통 설정: API Key 로드 + LLM 생성 함수

In [1]:
from dotenv import load_dotenv
import os
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# 사용할 제공자의 키만 있으면 됨 — 설정 여부와 앞 2글자만 확인용으로 출력 (키 전체는 출력하지 않음)
for name, key in (("GROQ_API_KEY", GROQ_API_KEY), ("OPENAI_API_KEY", OPENAI_API_KEY)):
    status = f"설정됨 ({key[:2]}...)" if key else "없음"
    print(f"{name}: {status}")

GROQ_API_KEY: 설정됨 (gs...)
OPENAI_API_KEY: 설정됨 (sk...)


In [2]:
from langchain_openai import ChatOpenAI

# 사용할 LLM 제공자 선택: "groq" 또는 "openai" — 이 값만 바꾸면 이후 모든 셀의 모델이 바뀜
PROVIDER = "groq"

# 제공자별 설정 (Groq는 OpenAI 호환 API라서 base_url만 다르고, OpenAI는 base_url 생략 시 기본 엔드포인트 사용)
LLM_CONFIGS = {
    "groq": {
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",  # Groq API 엔드포인트
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "api_key": OPENAI_API_KEY,
        "base_url": None,
        "model": "gpt-4o-mini",  # 대안: "gpt-4o"
    },
}
MAX_RETRIES = 5  # 503 등 일시 장애 자동 재시도 횟수


def get_llm(temperature: float = 0.5, seed: int | None = None, provider: str | None = None, verbose: bool = True, **kwargs) -> ChatOpenAI:
    """선택한 제공자(기본값: PROVIDER)의 ChatOpenAI 인스턴스를 생성합니다. verbose=True 이면 생성된 모델 정보를 출력합니다."""
    provider = provider or PROVIDER
    if provider not in LLM_CONFIGS:
        raise ValueError(f"지원하지 않는 제공자: {provider} (사용 가능: {list(LLM_CONFIGS)})")
    config = LLM_CONFIGS[provider]
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {provider.upper()}_API_KEY 가 없습니다.")
    llm = ChatOpenAI(
        **config,
        temperature=temperature,
        seed=seed,  # 값을 고정하면 재현성이 향상됨 (완전히 동일한 결과는 보장되지 않음)
        max_retries=MAX_RETRIES,
        **kwargs,
    )
    if verbose:
        # 생성된 모델 정보 출력 (API 키는 출력하지 않음)
        print(f"[LLM 생성] provider={provider} | model={llm.model_name}")
    return llm

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = get_llm(temperature=0.5)

# 1단계 프롬프트: 여행지 → 대표 명소 이름 1개
prompt1 = ChatPromptTemplate.from_messages([
    ("system", "당신은 세계 여행 전문가입니다. 질문한 여행지의 가장 대표적인 관광 명소 1곳의 이름만 답하세요. "
               "설명, 따옴표, 마침표 없이 명소 이름만 한국어로 출력하세요."),
    ("user", "여행지: {destination}"),
])

# 2단계 프롬프트: 명소 이름 → 상세 정보
prompt2 = ChatPromptTemplate.from_messages([
    ("system", "당신은 관광 명소 해설 가이드입니다. 명소에 대해 아래 형식으로 한국어로 설명하세요. "
               "각 항목은 번호를 붙여 2~3개씩 작성하세요.\n\n"
               "**명소:** (명소 이름)\n\n**역사:**\n1. ...\n\n**특징:**\n1. ...\n\n**방문 팁:**\n1. ..."),
    ("user", "{destination}에 있는 명소 '{place}'에 대해 알려주세요."),
])

# 각 단계 체인
chain1 = prompt1 | llm | StrOutputParser()   # 입력: {destination} → 출력: 명소 이름(str)
chain2 = prompt2 | llm | StrOutputParser()   # 입력: {destination, place} → 출력: 상세 정보(str)

[LLM 생성] provider=groq | model=openai/gpt-oss-120b


In [4]:
# 두 체인을 LCEL 로 연결
# RunnablePassthrough.assign 은 입력 dict 를 그대로 두고 새 키를 추가하므로
# 최종 결과 dict 에 destination / place(1단계) / detail(2단계) 이 모두 남아 각 단계를 확인할 수 있음
full_chain = (
    RunnablePassthrough.assign(place=chain1)      # 1단계: place 키 추가
    | RunnablePassthrough.assign(detail=chain2)   # 2단계: place 를 받아 detail 키 추가
)

result = full_chain.invoke({"destination": "로마"})

print(f"입력: {result['destination']}")
print(f"1단계 결과(추천 명소): {result['place']}")
print("\n2단계 결과(상세 정보):")
print(result["detail"])

입력: 로마
1단계 결과(추천 명소): 콜로세움

2단계 결과(상세 정보):
**명소:** 콜로세움  

**역사:**  
1. 서기 72년~80년 사이에 베스파시아누스 황제와 그의 아들 티투스 황제에 의해 건설되어, 고대 로마의 검투사 경기와 공개 처형 등 대규모 오락 행사의 중심지였다.  
2. 중세에는 성벽으로 활용되고, 이후 지진과 화재, 그리고 건축 자재 도난으로 상당히 훼손되었지만, 18세기부터는 유럽 각국의 여행가들이 “고대 로마의 상징”이라며 보존에 관심을 갖기 시작했다.  

**특징:**  
1. 원형 구조와 80개의 입구, 4층에 이르는 복합적인 아치형 설계는 당시 건축 기술의 정점을 보여준다.  
2. 약 5만 명을 수용할 수 있었으며, 무대 아래에는 해양 전투를 재현할 수 있는 복잡한 배수·수로 시스템이 있었다.  

**방문 팁:**  
1. 입장권은 온라인 사전 예매가 필수이며, 조조(오전 8시~10시) 입장은 비교적 한산해 사진 촬영에 유리하다.  
2. 가이드 오디오(다국어)와 함께 ‘지하 투어’를 예약하면 검투사들이 사용하던 지하 통로와 동물 우리를 직접 볼 수 있다.  
3. 주변에 위치한 로마 포럼과 팔라티노 언덕을 연계 티켓으로 구매하면 한 번에 세 곳을 편리하게 관람할 수 있다.


In [5]:
# 여러 여행지로 테스트 (batch 로 한 번에 실행)
destinations = ["뉴욕", "도쿄"]
results = full_chain.batch([{"destination": d} for d in destinations])

for r in results:
    print("=" * 60)
    print(f"입력: {r['destination']}")
    print(f"1단계 결과(추천 명소): {r['place']}")
    print("2단계 결과(상세 정보):")
    print(r["detail"])

입력: 뉴욕
1단계 결과(추천 명소): 자유의 여신상
2단계 결과(상세 정보):
**명소:** 자유의 여신상  

**역사:**  
1. 1875년 프랑스 조각가 프레데리크·오귀스트·바르톨디가 설계하고, 프랑스 정부가 미국 독립 100주년을 기념해 선물한 조형물이다.  
2. 1886년 10월 28일 뉴욕 항에 설치돼 개관했으며, 당시 세계에서 가장 높은 동상(93 m)으로 기록되었다.  
3. 1916년부터 1944년까지는 이민자들의 입국 심사 관문 역할을 했으며, 1984년에는 미국 국립기념물로 지정되었다.  

**특징:**  
1. 구리판을 겹쳐 만든 외피가 산화하면서 특유의 초록빛을 띠게 되었으며, 7개의 방사형 스파이크가 ‘빛을 비추는 횃불’ 형태를 만든다.  
2. 내부에는 354개의 계단이 있어, 관람객은 5층(왕관)까지 올라가 뉴욕 항과 맨해튼 스카이라인을 360도 파노라마로 감상할 수 있다.  
3. 매년 약 4백만 명이 방문하는 세계적인 관광 명소이자, 자유와 민주주의를 상징하는 국제적인 아이콘이다.  

**방문 팁:**  
1. 왕관 입장은 사전 예약이 필수이며, 티켓은 최소 2~3개월 전 온라인으로 구매하는 것이 안전하다.  
2. 자유의 여신상 방문은 배터리 파크(배터리 파크 시티)에서 출발하는 페리 이용이 일반적이며, 일찍 출발하면 대기 시간을 크게 줄일 수 있다.  
3. 날씨가 좋은 날에는 망원경을 챙겨 엘리스를 바라보며 맨해튼 스카이라인과 엘리스를 동시에 감상하면 더욱 풍성한 사진을 남길 수 있다.
입력: 도쿄
1단계 결과(추천 명소): 도쿄 타워
2단계 결과(상세 정보):
**명소:** 도쿄 타워  

**역사:**  
1. 1958년 12월 23일에 완공돼, 당시 세계에서 가장 높은 철탑(333 m)으로 세계 기록을 세웠다.  
2. 일본 전후 경제 성장기의 상징으로 건설됐으며, 전쟁 피해 복구와 관광 활성화를 목표로 했다.  
3. 1990년대 이후에는 전파 송신 기능 외에도 전망대와 레스토랑, 이벤트 공간 등 복합 문화